In [19]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from tqdm import tqdm

from MoE.mixture_of_experts import MoE

In [12]:
BATCH_SIZE = 128
NUM_CLASSES = 10
EPOCHS = 10
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.,), (1,))
])


train_dataset = datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
test_dataset  = datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader  = DataLoader(test_dataset, batch_size=BATCH_SIZE)

Files already downloaded and verified
Files already downloaded and verified


In [13]:
train_dataset[0][0].shape

torch.Size([3, 32, 32])

In [26]:
class CIFAR10Classifier(nn.Module):
    def __init__(self, moe_dim=128, num_experts=10, num_classes=10):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, stride=2, padding=1),  # [B, 32, 16, 16]
            nn.ReLU(),
            nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1), # [B, 64, 8, 8]
            nn.ReLU(),
            nn.Conv2d(64, 128, kernel_size=3, stride=2, padding=1), # [B, 128, 4, 4]
            nn.ReLU(),
        )
        self.flatten = nn.Flatten()  # [B, 64*8*8 = 4096]
        self.proj = nn.Linear(128 * 4 * 4, moe_dim)
        self.moe = MoE(dim=moe_dim, num_experts=num_experts, hidden_dim=moe_dim * 4, activation=nn.ReLU)
        self.classifier = nn.Linear(moe_dim, num_classes)

    def forward(self, x):
        x = self.conv(x)         # [B, C, H, W]
        x = self.flatten(x)      # [B, D_flat]
        x = self.proj(x)         # [B, D]
        x = x.unsqueeze(1)       # [B, 1, D] — как требует MoE
        x, moe_loss = self.moe(x)
        x = x.squeeze(1)         # [B, D]
        logits = self.classifier(x)
        return logits, moe_loss

In [27]:
model = CIFAR10Classifier().to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

for epoch in range(EPOCHS):
    model.train()
    total_loss, correct = 0, 0

    for imgs, labels in tqdm(train_loader):
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()

        logits, moe_loss = model(imgs)
        loss = criterion(logits, labels) + moe_loss
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * imgs.size(0)
        correct += (logits.argmax(dim=1) == labels).sum().item()

    acc = correct / len(train_dataset)
    print(f"Epoch {epoch+1}: Loss={total_loss/len(train_dataset):.4f}, Acc={acc:.4f}")

  0%|          | 0/391 [00:00<?, ?it/s]

100%|██████████| 391/391 [00:34<00:00, 11.37it/s]


Epoch 1: Loss=2.3136, Acc=0.0987


100%|██████████| 391/391 [00:33<00:00, 11.55it/s]


Epoch 2: Loss=2.3130, Acc=0.1009


...

Epoch 8: Loss=2.3130, Acc=0.0983


100%|██████████| 391/391 [00:33<00:00, 11.52it/s]


Epoch 9: Loss=2.3130, Acc=0.0978


100%|██████████| 391/391 [00:34<00:00, 11.34it/s]

Epoch 10: Loss=2.3130, Acc=0.0967


In [2]:
from fastmoe.fmoe.megatron import fmoefy
model = fmoefy(model, fmoe_num_experts=4)

ModuleNotFoundError: No module named 'fmoe_cuda'